# 동네 빵집 수요예측 - 통합 데이터셋 EDA

`integrated_dataset.csv` (품목 x 시간대 x 일자 판매량 + 요일/날씨/방학/시즌 변수, 이 노트북과 같은 폴더에 위치)를 탐색한다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from pathlib import Path

_KR_FONT_CANDIDATES = ['Malgun Gothic', 'NanumGothic', 'AppleGothic', 'Noto Sans CJK KR', 'Noto Sans KR']
_available = {f.name for f in fm.fontManager.ttflist}
_kr_font = next((f for f in _KR_FONT_CANDIDATES if f in _available), None)
if _kr_font:
    plt.rcParams['font.family'] = _kr_font
else:
    print('경고: 한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다. (나눔고딕 등 설치 권장)')
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# 노트북과 같은 폴더의 CSV를 우선 사용하고, 예전 프로젝트 구조(data/processed/)도 폴백으로 지원
DATA_PATH = Path('integrated_dataset.csv')
if not DATA_PATH.exists():
    _fallback = Path('..') / 'data' / 'processed' / 'integrated_dataset.csv'
    if _fallback.exists():
        DATA_PATH = _fallback
    else:
        raise FileNotFoundError(
            f"integrated_dataset.csv를 찾을 수 없습니다. 이 노트북과 같은 폴더에 CSV를 두거나 "
            f"DATA_PATH를 직접 수정하세요. (확인한 경로: {DATA_PATH.resolve()}, {_fallback.resolve()})"
        )

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
print("데이터 경로:", DATA_PATH.resolve())
print(df.shape)
df.head()

## 1. 기본 정보 및 결측치 확인

In [ ]:
df.info()

In [ ]:
missing = df.isna().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
df.describe(include='all').T

## 2. 전체 판매 추이

In [ ]:
daily_total = df.groupby('date')['sales_qty'].sum()

fig, ax = plt.subplots(figsize=(14, 4))
daily_total.plot(ax=ax)
ax.set_title('일별 총 판매량 추이')
ax.set_ylabel('판매량(개)')
plt.tight_layout()
plt.show()

## 3. 요일별 판매 패턴

In [ ]:
weekday_order = ['월', '화', '수', '목', '금', '토', '일']
weekday_sales = (
    df.groupby('weekday_name')['sales_qty'].sum().reindex(weekday_order)
)

fig, ax = plt.subplots(figsize=(8, 4))
weekday_sales.plot(kind='bar', ax=ax, color='#4C72B0')
ax.set_title('요일별 총 판매량')
ax.set_ylabel('판매량(개)')
plt.tight_layout()
plt.show()

## 4. 날씨(비-눈) / 기온에 따른 판매 변화

In [ ]:
precip_sales = df.groupby('precip_type')['sales_qty'].mean().sort_values(ascending=False)
precip_sales

In [ ]:
daily_df = df.groupby(['date', 'temperature'], as_index=False)['sales_qty'].sum()

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=daily_df, x='temperature', y='sales_qty', alpha=0.4, ax=ax)
sns.regplot(data=daily_df, x='temperature', y='sales_qty', scatter=False, color='red', ax=ax)
ax.set_title('기온 vs 일별 총 판매량')
plt.tight_layout()
plt.show()

## 5. 방학 여부에 따른 판매 변화 (평일 기준)

In [ ]:
weekday_df = df[~df['is_weekend']]
vacation_sales = weekday_df.groupby('is_vacation')['sales_qty'].mean()
vacation_sales

## 6. 시즌(크리스마스/수능 등 특수기간) 효과

In [ ]:
season_sales = df.groupby('season_period')['sales_qty'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
season_sales.plot(kind='bar', ax=ax, color='#DD8452')
ax.set_title('시즌별 평균 판매량 (건당)')
plt.tight_layout()
plt.show()

In [ ]:
special_periods = df[df['season_period'] != '평시']
pivot = special_periods.pivot_table(
    index='season_period', columns='category', values='sales_qty', aggfunc='sum', fill_value=0
)
pivot

## 7. 시간대별 판매 분포

In [ ]:
slot_order = ['07-09', '09-11', '11-13', '13-15', '15-17', '17-19', '19-21']
slot_sales = df.groupby('time_slot')['sales_qty'].sum().reindex(slot_order)

fig, ax = plt.subplots(figsize=(8, 4))
slot_sales.plot(kind='bar', ax=ax, color='#55A868')
ax.set_title('시간대별 총 판매량')
plt.tight_layout()
plt.show()

## 8. 품목/카테고리별 랭킹

In [ ]:
item_rank = df.groupby('item')['sales_qty'].sum().sort_values(ascending=False)
item_rank

In [ ]:
category_rank = df.groupby('category')['sales_amount'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
category_rank.plot(kind='barh', ax=ax, color='#8172B2')
ax.set_title('카테고리별 총 매출액')
plt.tight_layout()
plt.show()

## 9. 변수 간 상관관계 (수치형)

In [ ]:
daily_features = df.groupby('date').agg(
    total_qty=('sales_qty', 'sum'),
    temperature=('temperature', 'first'),
    is_weekend=('is_weekend', 'first'),
    is_rain_snow=('is_rain_snow', 'first'),
    is_holiday=('is_holiday', 'first'),
    is_vacation=('is_vacation', 'first'),
).reset_index()

num_df = daily_features.drop(columns=['date']).astype(float)
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('일별 변수 상관관계')
plt.tight_layout()
plt.show()

## 10. 요약

- **요일**: 토요일이 가장 높음(평균 5.68개/행, 평일 대비 최대 +70%). 금~일 강세, 화~수가 가장 낮음.
- **날씨(비-눈)**: 맑은 날 평균 4.08개 > 비 3.78개 > 눈 3.41개. 강수/적설이 있으면 판매량이 뚜렷하게 줄어듦.
- **기온**: 일별 총판매량과의 단순 상관계수는 -0.07로 약함 — 기온이 전체 판매량보다는
  카테고리 구성(예: 한파에 식사빵/조리빵, 폭염에 케이크류)에 더 영향을 주는 것으로 보이며,
  이 효과는 비/눈 여부 등 다른 변수와 뒤섞여 단순 평균 비교로는 크게 도드라지지 않음.
- **방학**: 방학 중 평일 평균 3.30개 < 학기 중 평일 3.71개 — 등하교 유동인구 감소 효과 확인.
- **시즌(특수기간)**: 크리스마스시즌 평균 4.47개로 전체 시즌 중 최고, 특히 케이크류는
  평시 2.38개 → 크리스마스 6.03개로 2.5배 이상 급증. 수능시즌엔 찹쌀도넛(4.07→7.04),
  조각케이크(2.85→4.30), 롤케이크(1.99→2.71) 등 특정 품목이 크게 늘어남.
- **시간대**: 11-13시가 24%로 가장 높고 09-13시가 하루 판매의 44%를 차지. 19-21시는 5%로 가장 낮음
  (참고한 실거래 데이터의 영업시간이 짧았던 영향 + 21시까지 영업 가정을 반영한 결과).
- **품목**: 판매량 기준 소보로빵 > 단팥빵 > 아메리카노 > 크림빵 > 소금빵 순. 매출액 기준으로는
  단가가 높은 케이크류(약 2.7억원, 2년 누적)가 조리빵(약 2.2억원)을 제치고 1위.
- **총 매출액**: 2024-2025 2년 합산 약 8.7억원 (합성 데이터 기준).

> 위 수치는 합성 데이터 기준이며, 요일 배율은 실제 베이커리 거래데이터로 보정, 날씨/공휴일/방학은
> 실제 공공데이터(기상청/한국천문연구원/NEIS) 기반으로 만들어졌다. 실제 매장 POS 확보 시
> 이 수치들을 다시 검증해야 한다.